# Notebook 11 — CI/CD for Genie Spaces

**What you’ll learn:** How to promote Genie spaces across environments (dev → staging → prod) using code, not manual clicks.

**Why CI/CD for Genie?** In production, you don’t want to manually recreate spaces. You want to:
- **Export** a space definition from dev (JSON)
- **Transform** it (swap catalog/schema references for the target environment)
- **Deploy** it to prod via API

This is the same export → transform → deploy pattern used for dashboards and pipelines.

**This notebook is optional** — it’s for teams ready to operationalize their Genie spaces. The concepts here apply whether you use GitHub Actions, Azure DevOps, or Databricks Asset Bundles (DABs).

**Before you start:** Run notebook **03** (Genie spaces).

**Compute:** Serverless.

## Current state (April 2026)

| Mechanism | Status | Notes |
|-----------|--------|-------|
| **Genie CRUD REST APIs** | **Public Preview** | `POST`, `GET`, `PATCH` on `/api/2.0/genie/spaces` — the supported CI/CD path today |
| **Databricks Asset Bundles (DABs)** | In progress (DIRECT mode PR) | `databricks/cli` PR #4191 adds `genie_spaces` resource — not yet GA |
| **Terraform provider** | Not yet available | No Genie resource exposed; on roadmap after DABs |

## The 3-phase promotion pattern

```
┌──────────┐     ┌─────────────┐     ┌──────────┐
│  EXPORT  │ ──> │  TRANSFORM  │ ──> │  DEPLOY  │
│  (dev)   │     │  (CI / Git) │     │  (prod)  │
└──────────┘     └─────────────┘     └──────────┘
```

1. **Export:** `GET /api/2.0/genie/spaces/{id}?include_serialized_space=true`
2. **Transform:** Remap `catalog.schema` references in `serialized_space`, commit JSON to Git
3. **Deploy:** `POST` (create) or `PATCH` (update) to the target workspace

**Key gotcha:** Table identifiers inside `serialized_space` are hardcoded as `catalog.schema.table`. When promoting across environments with different catalogs, you **must** remap these references.

## Load config

In [ ]:
%run ./00_workshop_config

In [ ]:
import re, json, copy, requests
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()
host_url = w.config.host.rstrip("/")
headers = {**w.config.authenticate(), "Content-Type": "application/json"}

_cfg = spark.table(full_table("workshop_config")).toPandas().set_index("key")["value"].to_dict()
GENIE_SPACE_ID = _cfg.get("genie_space_id", "")
print(f"Space: {GENIE_SPACE_ID}")


## Phase 1 — Export

Fetch the full space configuration including `serialized_space`.

In [ ]:
export_resp = requests.get(
    f"{host_url}/api/2.0/genie/spaces/{GENIE_SPACE_ID}?include_serialized_space=true",
    headers=headers,
)
export_resp.raise_for_status()
exported = export_resp.json()
_ss = exported.get("serialized_space")
space_def = json.loads(_ss) if isinstance(_ss, str) else (_ss if isinstance(_ss, dict) else {})

print(f"Title: {exported.get('title') or exported.get('display_name')}")
print(f"Tables: {len(space_def.get('data_sources', {}).get('tables', []))}")
print(f"Curated SQL: {len(space_def.get('instructions', {}).get('example_question_sqls', []))}")
print(f"Sample questions: {len(space_def.get('config', {}).get('sample_questions', []))}")


## Phase 2 — Transform (remap catalog/schema)

Every table identifier and SQL reference must point at the target environment.

In [ ]:
SOURCE_FQN = f"{CATALOG}.{SCHEMA}"
TARGET_FQN = f"{CATALOG}_prod.{SCHEMA}"

def remap_references(space_dict, source_fqn, target_fqn):
    raw = json.dumps(space_dict)
    return json.loads(raw.replace(source_fqn, target_fqn))

prod_space_def = remap_references(copy.deepcopy(space_def), SOURCE_FQN, TARGET_FQN)

dev_tables = [t["identifier"] for t in space_def.get("data_sources", {}).get("tables", [])]
prod_tables = [t["identifier"] for t in prod_space_def.get("data_sources", {}).get("tables", [])]
print("DEV:", dev_tables[:3])
print("PROD:", prod_tables[:3])


## Phase 3 — Deploy (demo: create a copy in the same workspace)

In production you would target a different workspace. Here we create a `[CICD]` copy.

In [ ]:
import html as _html

_wh_list = list(w.warehouses.list())
WAREHOUSE_ID = None
for _wh in _wh_list:
    _st = str(_wh.state).upper() if _wh.state else ""
    if _st in ("RUNNING", "STARTING") or getattr(_wh, "enable_serverless_compute", False):
        WAREHOUSE_ID = str(_wh.id)
        break
if not WAREHOUSE_ID and _wh_list:
    WAREHOUSE_ID = str(_wh_list[0].id)

if WAREHOUSE_ID:
    _title = exported.get("title") or exported.get("display_name") or "Manufacturing Analytics"
    _demo_title = f"[CICD DEMO] {_title}"

    # Reuse existing CI/CD demo space if one already exists with the same title
    new_id = None
    _existing = requests.get(f"{host_url}/api/2.0/genie/spaces", headers=headers)
    if _existing.status_code == 200:
        for _s in _existing.json().get("spaces", []):
            if _s.get("title") == _demo_title:
                new_id = _s.get("space_id") or _s.get("id")
                print(f"Already exists: {_demo_title!r} -> {new_id}")
                # PATCH the existing space with the latest exported content
                _pr = requests.patch(
                    f"{host_url}/api/2.0/genie/spaces/{new_id}",
                    headers=headers,
                    json={
                        "title": _demo_title,
                        "description": f"CI/CD deployed copy. Source: {GENIE_SPACE_ID}",
                        "warehouse_id": WAREHOUSE_ID,
                        "serialized_space": json.dumps(space_def),
                    },
                )
                if _pr.status_code == 200:
                    print(f"  Updated space content from dev export.")
                else:
                    print(f"  PATCH returned {_pr.status_code}: {_pr.text[:200]}")
                break

    if not new_id:
        deploy_resp = requests.post(
            f"{host_url}/api/2.0/genie/spaces",
            headers=headers,
            json={
                "title": _demo_title,
                "description": f"CI/CD deployed copy. Source: {GENIE_SPACE_ID}",
                "warehouse_id": WAREHOUSE_ID,
                "serialized_space": json.dumps(space_def),
            },
        )
        if deploy_resp.status_code == 200:
            new_id = deploy_resp.json().get("space_id") or deploy_resp.json().get("id")
            print(f"Created: {new_id}")
        else:
            print(f"Deploy failed ({deploy_resp.status_code}): {deploy_resp.text[:300]}")

    if new_id:
        m = re.search(r"adb-(\d+)\.", host_url)
        o = m.group(1) if m else ""
        q = f"?o={o}" if o else ""
        url = f"{host_url}/genie/rooms/{new_id}{q}"
        displayHTML(f'<a href="{_html.escape(url, quote=True)}" target="_blank" rel="noopener">Open CI/CD deployed space</a>')
else:
    print("No SQL warehouse found. Export and transform steps above are the portable parts.")


## PATCH pattern (update existing prod space)

When promoting **changes** to a space that already exists, use the round-trip update:

```python
# 1. Fetch existing prod space
prod_resp = requests.get(f"{host}/api/2.0/genie/spaces/{PROD_ID}?include_serialized_space=true", headers=headers)
prod_existing = json.loads(prod_resp.json()["serialized_space"])

# 2. Apply targeted changes from dev export
prod_existing["instructions"] = dev_space_def["instructions"]

# 3. Sort example_question_sqls by id (API requirement)
eqs = prod_existing.get("instructions", {}).get("example_question_sqls", [])
eqs.sort(key=lambda x: x.get("id", ""))

# 4. PATCH back
requests.patch(f"{host}/api/2.0/genie/spaces/{PROD_ID}", headers=headers,
               json={"serialized_space": json.dumps(prod_existing)})
```

Always fetch first, apply targeted changes, then PATCH. This preserves opaque IDs and server-managed fields.

## Reference architecture

```
┌─────────────────────────────────────────────┐
│               Git Repository              │
│  genie_spaces/                             │
│    manufacturing_quality.json               │
│    transform.py                             │
│    deploy.py                                │
└────────────────────────┬────────────────────┘
                         │ PR merge triggers CI
                         ▼
┌─────────────────────────────────────────────┐
│         GitHub Actions / Azure DevOps       │
│  1. Checkout repo                           │
│  2. transform.py (remap dev → prod)         │
│  3. deploy.py (POST or PATCH)               │
│  4. Run benchmarks (notebook 07 via Job)     │
│  5. Pass → done. Fail → alert + rollback.   │
└─────────────────────────────────────────────┘
```

### Field-built tools (open source)

| Tool | Author | What it does |
|------|--------|-------------|
| [SpaceOps CLI](https://github.com/charotAmine/databricks-spaceops) | Amine Charot | YAML-driven config, snapshot + deploy across environments |
| [genie-space-cicd](https://github.com/mohade09/genie-space-cicd) | Debadatta Mohapatra | Sample CI/CD repo for Genie spaces |
| [genie-migration-tool](https://github.com/reynoldspravindev/databricks-genie-migration-tool) | Reynolds Pravindev | CLI wrapper for CRUD APIs |

### Further reading

- [Azure DevOps + Databricks CI/CD](https://learn.microsoft.com/en-us/azure/databricks/dev-tools/ci-cd/ci-cd-azure-devops)
- [GitHub Actions for Databricks](https://docs.databricks.com/en/dev-tools/ci-cd/ci-cd-github.html)
- [Genie REST API reference](https://docs.databricks.com/api/workspace/genie)
- DABs native support: Expected in a future release (DIRECT mode).